In [5]:
!pip install pandas nltk scikit-learn sacrebleu


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install seaborn matplotlib


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import json
import re
import ast
from typing import List, Dict, Optional, Tuple
from collections import Counter
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

In [8]:
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("✓ Importazioni completate")

✓ Importazioni completate


In [9]:
# Configurazione per ordine invertito
MODEL_NAME = "granite3.1-moe:3b"
MODEL_NAME_CLEAN = MODEL_NAME.replace(':', '_')
BASE_PATH = f"../Output Performance Degradation Analysis/{MODEL_NAME_CLEAN}_six_task_invert"

# Task configurati - ORDINE INVERTITO
TASK_NAMES = {
    1: "json_only",
    2: "json_sentiment", 
    3: "json_sentiment_emotion",
    4: "json_sentiment_emotion_topic",
    5: "json_sentiment_emotion_topic_ner",
    6: "json_sentiment_emotion_topic_ner_translation"
}

# Campi richiesti per ogni task - ORDINE INVERTITO
REQUIRED_FIELDS = {
    1: ["review"],
    2: ["review", "sentiment"],
    3: ["review", "sentiment", "emotion"],
    4: ["review", "sentiment", "emotion", "topic"],
    5: ["review", "sentiment", "emotion", "topic", "entities"],
    6: ["review", "sentiment", "emotion", "topic", "entities", "translate"]
}

print(f"✓ Configurazione completata:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - Percorso base: {BASE_PATH}")
print(f"  - Task configurati: {len(TASK_NAMES)}")

✓ Configurazione completata:
  - Modello: granite3.1-moe:3b
  - Percorso base: ../Output Performance Degradation Analysis/granite3.1-moe_3b_six_task_invert
  - Task configurati: 6


In [10]:
def is_valid_json_format(json_string: str, required_fields: List[str]) -> bool:
    """
    Verifica se il JSON è formalmente valido e ha tutti i campi richiesti.
    
    Args:
        json_string (str): Il JSON da validare
        required_fields (List[str]): I campi richiesti
        
    Returns:
        bool: True se valido, False altrimenti
    """
    if pd.isna(json_string) or json_string is None:
        return False
    
    try:
        # Parse del JSON
        json_obj = json.loads(json_string)
        
        # Verifica che sia un dizionario
        if not isinstance(json_obj, dict):
            return False
        
        # Verifica che abbia tutti i campi richiesti
        json_fields = set(json_obj.keys())
        required_fields_set = set(required_fields)
        
        # Deve avere tutti i campi richiesti e nessun campo extra
        if json_fields != required_fields_set:
            return False
            
        return True
        
    except (json.JSONDecodeError, TypeError):
        return False

In [11]:
def calculate_bleu_score(reference: str, candidate: str) -> float:
    """
    Calcola il BLEU score tra reference e candidate.
    """
    if pd.isna(reference) or pd.isna(candidate) or reference == "" or candidate == "":
        return 0.0
    
    try:
        ref_tokens = nltk.word_tokenize(reference.lower())
        cand_tokens = nltk.word_tokenize(candidate.lower())
        
        smoothing = SmoothingFunction()
        score = sentence_bleu([ref_tokens], cand_tokens, 
                            smoothing_function=smoothing.method1)
        
        return score * 100
        
    except Exception:
        return 0.0

In [12]:
def exact_match_classification(reference: str, candidate: str) -> bool:
    """
    Verifica exact match per classificazione (sentiment, emotion, topic).
    """
    try:
        if reference is None or candidate is None:
            return False
        
        ref_str = str(reference)
        cand_str = str(candidate)
        
        if ref_str.lower() in ['nan', 'none', ''] or cand_str.lower() in ['nan', 'none', '']:
            return False
        
        return ref_str.lower().strip() == cand_str.lower().strip()
        
    except Exception:
        return False


In [13]:
def parse_entities(entities_str: str) -> List[Dict]:
    """
    Parsifica la stringa delle entità in una lista di dizionari.
    """
    if pd.isna(entities_str) or entities_str == "":
        return []
    
    try:
        if entities_str.startswith('['):
            return json.loads(entities_str)
        
        return ast.literal_eval(entities_str)
        
    except (json.JSONDecodeError, ValueError, SyntaxError):
        return []

In [14]:
def calculate_ner_f1(reference_entities: str, candidate_entities: List[Dict]) -> float:
    """
    Calcola F1 score per NER.
    """
    ref_entities = parse_entities(reference_entities)
    
    if not ref_entities and not candidate_entities:
        return 100.0
    
    if not ref_entities or not candidate_entities:
        return 0.0
    
    ref_set = set()
    for entity in ref_entities:
        if isinstance(entity, dict) and 'label' in entity and 'value' in entity:
            if entity['value'] is not None:
                ref_set.add((entity['label'], str(entity['value']).lower().strip()))
    
    cand_set = set()
    for entity in candidate_entities:
        if isinstance(entity, dict) and 'label' in entity and 'value' in entity:
            if entity['value'] is not None:
                cand_set.add((entity['label'], str(entity['value']).lower().strip()))
    
    if not ref_set and not cand_set:
        return 100.0
    
    if not ref_set or not cand_set:
        return 0.0
    
    intersection = ref_set.intersection(cand_set)
    
    if len(intersection) == 0:
        return 0.0
    
    precision = len(intersection) / len(cand_set)
    recall = len(intersection) / len(ref_set)
    
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1 * 100

In [15]:
def evaluate_task_file(task_number: int) -> pd.DataFrame:
    """
    Valuta un singolo file di task e calcola tutte le metriche.
    """
    task_name = TASK_NAMES[task_number]
    required_fields = REQUIRED_FIELDS[task_number]
    
    input_file = f"{BASE_PATH}_cleaned/task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}_cleaned.csv"
    
    print(f"Valutando Task {task_number} ({task_name})...")
    
    if not os.path.exists(input_file):
        print(f"❌ File non trovato: {input_file}")
        return pd.DataFrame()
    
    df = pd.read_csv(input_file)
    print(f"✓ File caricato: {len(df)} righe")
    
    # Inizializza colonne metriche
    df['json_valid'] = 0
    df['bleu_review'] = 0.0
    df['bleu_translate'] = 0.0
    df['em_sentiment'] = 0
    df['em_emotion'] = 0
    df['em_topic'] = 0
    df['f1_ner'] = 0.0
    
    # Calcola metriche per ogni riga
    for idx, row in df.iterrows():
        json_output = row.get('json_output', None)
        
        # 1. Validazione JSON
        df.at[idx, 'json_valid'] = 1 if is_valid_json_format(json_output, required_fields) else 0
        
        if df.at[idx, 'json_valid'] == 0:
            continue
        
        try:
            json_obj = json.loads(json_output)
            
            # 2. BLEU su review (SEMPRE PRESENTE IN TUTTI I TASK)
            ref_review = row.get('review', '')
            cand_review = json_obj.get('review', '')
            df.at[idx, 'bleu_review'] = calculate_bleu_score(ref_review, cand_review)
            
            # 3. EM su sentiment (se richiesto)
            if 'sentiment' in required_fields:
                ref_sentiment = row.get('sentiment', '')
                cand_sentiment = json_obj.get('sentiment', '')
                df.at[idx, 'em_sentiment'] = 1 if exact_match_classification(ref_sentiment, cand_sentiment) else 0
            
            # 4. EM su emotion (se richiesto)
            if 'emotion' in required_fields:
                ref_emotion = row.get('emotion_classification', '')
                cand_emotion = json_obj.get('emotion', '')
                df.at[idx, 'em_emotion'] = 1 if exact_match_classification(ref_emotion, cand_emotion) else 0
            
            # 5. EM su topic (se richiesto)
            if 'topic' in required_fields:
                ref_topic = row.get('topic_classification', '')
                cand_topic = json_obj.get('topic', '')
                df.at[idx, 'em_topic'] = 1 if exact_match_classification(ref_topic, cand_topic) else 0
            
            # 6. F1 su NER (se richiesto)
            if 'entities' in required_fields:
                ref_entities = row.get('entities', '')
                cand_entities = json_obj.get('entities', [])
                df.at[idx, 'f1_ner'] = calculate_ner_f1(ref_entities, cand_entities)
            
            # 7. BLEU su translate (se richiesto)
            if 'translate' in required_fields:
                ref_translate = row.get('translate', '')
                cand_translate = json_obj.get('translate', '')
                df.at[idx, 'bleu_translate'] = calculate_bleu_score(ref_translate, cand_translate)
                
        except (json.JSONDecodeError, TypeError):
            pass
    
    return df

In [16]:
# CARICAMENTO E VALUTAZIONE TASK
print("=" * 60)
print("CARICAMENTO E VALUTAZIONE TASK")
print("=" * 60)

# Dizionario per memorizzare i risultati
task_results = {}

for task_number in TASK_NAMES.keys():
    task_results[task_number] = evaluate_task_file(task_number)
    print("-" * 40)

print("✅ Valutazione completata per tutti i task!")

CARICAMENTO E VALUTAZIONE TASK
Valutando Task 1 (json_only)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 2 (json_sentiment)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 3 (json_sentiment_emotion)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 4 (json_sentiment_emotion_topic)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 5 (json_sentiment_emotion_topic_ner)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 6 (json_sentiment_emotion_topic_ner_translation)...
✓ File caricato: 500 righe
----------------------------------------
✅ Valutazione completata per tutti i task!


In [17]:
# STEP 1: JSON FORMAT + BLEU REVIEW
print("=" * 60)
print("STEP 1: JSON FORMAT + BLEU REVIEW")
print("=" * 60)

step1_results = {}

for task_number in TASK_NAMES.keys():
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    valid_json_df = df[df['json_valid'] == 1]
    
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_review_mean = valid_json_df['bleu_review'].mean() if len(valid_json_df) > 0 else 0.0
    
    step1_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_review': bleu_review_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Review: {bleu_review_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 1:")
for task_num, results in step1_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU Review = {results['bleu_review']:.2f}%")

STEP 1: JSON FORMAT + BLEU REVIEW
Task 1 (json_only):
  - JSON Exact Match: 98.00%
  - BLEU Review: 51.42% (su 490 JSON validi)
  - Campioni totali: 500
Task 2 (json_sentiment):
  - JSON Exact Match: 98.40%
  - BLEU Review: 66.91% (su 492 JSON validi)
  - Campioni totali: 500
Task 3 (json_sentiment_emotion):
  - JSON Exact Match: 99.00%
  - BLEU Review: 76.41% (su 495 JSON validi)
  - Campioni totali: 500
Task 4 (json_sentiment_emotion_topic):
  - JSON Exact Match: 98.20%
  - BLEU Review: 80.48% (su 491 JSON validi)
  - Campioni totali: 500
Task 5 (json_sentiment_emotion_topic_ner):
  - JSON Exact Match: 98.40%
  - BLEU Review: 85.07% (su 492 JSON validi)
  - Campioni totali: 500
Task 6 (json_sentiment_emotion_topic_ner_translation):
  - JSON Exact Match: 58.80%
  - BLEU Review: 78.37% (su 294 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 1:
Task 1: JSON EM = 98.00%, BLEU Review = 51.42%
Task 2: JSON EM = 98.40%, BLEU 

In [18]:
# STEP 2: JSON + BLEU REVIEW + SENTIMENT
print("=" * 60)
print("STEP 2: JSON + BLEU REVIEW + SENTIMENT")
print("=" * 60)

step2_results = {}

for task_number in [2, 3, 4, 5, 6]:  # Task con sentiment
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_review_mean = valid_json_df['bleu_review'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    
    step2_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_review': bleu_review_mean,
        'em_sentiment': em_sentiment_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Review: {bleu_review_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 2:")
for task_num, results in step2_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU Review = {results['bleu_review']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%")


STEP 2: JSON + BLEU REVIEW + SENTIMENT
Task 2 (json_sentiment):
  - JSON Exact Match: 98.40%
  - BLEU Review: 66.91% (su 492 JSON validi)
  - EM Sentiment: 86.18% (su 492 JSON validi)
  - Campioni totali: 500
Task 3 (json_sentiment_emotion):
  - JSON Exact Match: 99.00%
  - BLEU Review: 76.41% (su 495 JSON validi)
  - EM Sentiment: 85.86% (su 495 JSON validi)
  - Campioni totali: 500
Task 4 (json_sentiment_emotion_topic):
  - JSON Exact Match: 98.20%
  - BLEU Review: 80.48% (su 491 JSON validi)
  - EM Sentiment: 86.15% (su 491 JSON validi)
  - Campioni totali: 500
Task 5 (json_sentiment_emotion_topic_ner):
  - JSON Exact Match: 98.40%
  - BLEU Review: 85.07% (su 492 JSON validi)
  - EM Sentiment: 82.32% (su 492 JSON validi)
  - Campioni totali: 500
Task 6 (json_sentiment_emotion_topic_ner_translation):
  - JSON Exact Match: 58.80%
  - BLEU Review: 78.37% (su 294 JSON validi)
  - EM Sentiment: 86.05% (su 294 JSON validi)
  - Campioni totali: 500
-----------------------------------------

In [19]:
# STEP 3: JSON + BLEU REVIEW + SENTIMENT + EMOTION
print("=" * 60)
print("STEP 3: JSON + BLEU REVIEW + SENTIMENT + EMOTION")
print("=" * 60)

step3_results = {}

for task_number in [3, 4, 5, 6]:  # Task con emotion
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_review_mean = valid_json_df['bleu_review'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    
    step3_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_review': bleu_review_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Review: {bleu_review_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 3:")
for task_num, results in step3_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU Review = {results['bleu_review']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%")


STEP 3: JSON + BLEU REVIEW + SENTIMENT + EMOTION
Task 3 (json_sentiment_emotion):
  - JSON Exact Match: 99.00%
  - BLEU Review: 76.41% (su 495 JSON validi)
  - EM Sentiment: 85.86% (su 495 JSON validi)
  - EM Emotion: 16.57% (su 495 JSON validi)
  - Campioni totali: 500
Task 4 (json_sentiment_emotion_topic):
  - JSON Exact Match: 98.20%
  - BLEU Review: 80.48% (su 491 JSON validi)
  - EM Sentiment: 86.15% (su 491 JSON validi)
  - EM Emotion: 5.30% (su 491 JSON validi)
  - Campioni totali: 500
Task 5 (json_sentiment_emotion_topic_ner):
  - JSON Exact Match: 98.40%
  - BLEU Review: 85.07% (su 492 JSON validi)
  - EM Sentiment: 82.32% (su 492 JSON validi)
  - EM Emotion: 3.25% (su 492 JSON validi)
  - Campioni totali: 500
Task 6 (json_sentiment_emotion_topic_ner_translation):
  - JSON Exact Match: 58.80%
  - BLEU Review: 78.37% (su 294 JSON validi)
  - EM Sentiment: 86.05% (su 294 JSON validi)
  - EM Emotion: 4.76% (su 294 JSON validi)
  - Campioni totali: 500
----------------------------

In [20]:
# STEP 4: JSON + BLEU REVIEW + SENTIMENT + EMOTION + TOPIC
print("=" * 60)
print("STEP 4: JSON + BLEU REVIEW + SENTIMENT + EMOTION + TOPIC")
print("=" * 60)

step4_results = {}

for task_number in [4, 5, 6]:  # Task con topic
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_review_mean = valid_json_df['bleu_review'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    em_topic_mean = valid_json_df['em_topic'].mean() * 100
    
    step4_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_review': bleu_review_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'em_topic': em_topic_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Review: {bleu_review_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Topic: {em_topic_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 4:")
for task_num, results in step4_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU Review = {results['bleu_review']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%, Topic EM = {results['em_topic']:.2f}%")


STEP 4: JSON + BLEU REVIEW + SENTIMENT + EMOTION + TOPIC
Task 4 (json_sentiment_emotion_topic):
  - JSON Exact Match: 98.20%
  - BLEU Review: 80.48% (su 491 JSON validi)
  - EM Sentiment: 86.15% (su 491 JSON validi)
  - EM Emotion: 5.30% (su 491 JSON validi)
  - EM Topic: 10.79% (su 491 JSON validi)
  - Campioni totali: 500
Task 5 (json_sentiment_emotion_topic_ner):
  - JSON Exact Match: 98.40%
  - BLEU Review: 85.07% (su 492 JSON validi)
  - EM Sentiment: 82.32% (su 492 JSON validi)
  - EM Emotion: 3.25% (su 492 JSON validi)
  - EM Topic: 5.28% (su 492 JSON validi)
  - Campioni totali: 500
Task 6 (json_sentiment_emotion_topic_ner_translation):
  - JSON Exact Match: 58.80%
  - BLEU Review: 78.37% (su 294 JSON validi)
  - EM Sentiment: 86.05% (su 294 JSON validi)
  - EM Emotion: 4.76% (su 294 JSON validi)
  - EM Topic: 11.90% (su 294 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 4:
Task 4: JSON EM = 98.20%, BLEU Review 

In [21]:
# STEP 5: JSON + BLEU REVIEW + SENTIMENT + EMOTION + TOPIC + NER
print("=" * 60)
print("STEP 5: JSON + BLEU REVIEW + SENTIMENT + EMOTION + TOPIC + NER")
print("=" * 60)

step5_results = {}

for task_number in [5, 6]:  # Task con NER
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_review_mean = valid_json_df['bleu_review'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    em_topic_mean = valid_json_df['em_topic'].mean() * 100
    f1_ner_mean = valid_json_df['f1_ner'].mean()
    
    step5_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_review': bleu_review_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'em_topic': em_topic_mean,
        'f1_ner': f1_ner_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Review: {bleu_review_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Topic: {em_topic_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - F1 NER: {f1_ner_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 5:")
for task_num, results in step5_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU Review = {results['bleu_review']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%, Topic EM = {results['em_topic']:.2f}%, NER F1 = {results['f1_ner']:.2f}%")


STEP 5: JSON + BLEU REVIEW + SENTIMENT + EMOTION + TOPIC + NER
Task 5 (json_sentiment_emotion_topic_ner):
  - JSON Exact Match: 98.40%
  - BLEU Review: 85.07% (su 492 JSON validi)
  - EM Sentiment: 82.32% (su 492 JSON validi)
  - EM Emotion: 3.25% (su 492 JSON validi)
  - EM Topic: 5.28% (su 492 JSON validi)
  - F1 NER: 35.93% (su 492 JSON validi)
  - Campioni totali: 500
Task 6 (json_sentiment_emotion_topic_ner_translation):
  - JSON Exact Match: 58.80%
  - BLEU Review: 78.37% (su 294 JSON validi)
  - EM Sentiment: 86.05% (su 294 JSON validi)
  - EM Emotion: 4.76% (su 294 JSON validi)
  - EM Topic: 11.90% (su 294 JSON validi)
  - F1 NER: 34.34% (su 294 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 5:
Task 5: JSON EM = 98.40%, BLEU Review = 85.07%, Sentiment EM = 82.32%, Emotion EM = 3.25%, Topic EM = 5.28%, NER F1 = 35.93%
Task 6: JSON EM = 58.80%, BLEU Review = 78.37%, Sentiment EM = 86.05%, Emotion EM = 4.76%, Topic

In [22]:
# STEP 6: TUTTE LE METRICHE + TRANSLATION
print("=" * 60)
print("STEP 6: TUTTE LE METRICHE + TRANSLATION")
print("=" * 60)

step6_results = {}

for task_number in [6]:  # Solo task con translation
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_review_mean = valid_json_df['bleu_review'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    em_topic_mean = valid_json_df['em_topic'].mean() * 100
    f1_ner_mean = valid_json_df['f1_ner'].mean()
    bleu_translate_mean = valid_json_df['bleu_translate'].mean()
    
    step6_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_review': bleu_review_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'em_topic': em_topic_mean,
        'f1_ner': f1_ner_mean,
        'bleu_translate': bleu_translate_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Review: {bleu_review_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Topic: {em_topic_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - F1 NER: {f1_ner_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - BLEU Translate: {bleu_translate_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 6:")
for task_num, results in step6_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU Review = {results['bleu_review']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%, Topic EM = {results['em_topic']:.2f}%, NER F1 = {results['f1_ner']:.2f}%, BLEU Translate = {results['bleu_translate']:.2f}%")


STEP 6: TUTTE LE METRICHE + TRANSLATION
Task 6 (json_sentiment_emotion_topic_ner_translation):
  - JSON Exact Match: 58.80%
  - BLEU Review: 78.37% (su 294 JSON validi)
  - EM Sentiment: 86.05% (su 294 JSON validi)
  - EM Emotion: 4.76% (su 294 JSON validi)
  - EM Topic: 11.90% (su 294 JSON validi)
  - F1 NER: 34.34% (su 294 JSON validi)
  - BLEU Translate: 9.66% (su 294 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 6:
Task 6: JSON EM = 58.80%, BLEU Review = 78.37%, Sentiment EM = 86.05%, Emotion EM = 4.76%, Topic EM = 11.90%, NER F1 = 34.34%, BLEU Translate = 9.66%


In [23]:
# SALVATAGGIO RISULTATI
print("=" * 60)
print("SALVATAGGIO RISULTATI")
print("=" * 60)

# Crea un DataFrame riassuntivo
summary_data = []

# Aggiungi risultati di tutti gli step
for step_name, step_results in [
    ("Step 1", step1_results),
    ("Step 2", step2_results), 
    ("Step 3", step3_results),
    ("Step 4", step4_results),
    ("Step 5", step5_results),
    ("Step 6", step6_results)
]:
    for task_num, results in step_results.items():
        row = {
            'step': step_name,
            'task_number': task_num,
            'task_name': results['task_name'],
            'json_em': results['json_em'],
            'bleu_review': results.get('bleu_review', None),
            'em_sentiment': results.get('em_sentiment', None),
            'em_emotion': results.get('em_emotion', None),
            'em_topic': results.get('em_topic', None),
            'f1_ner': results.get('f1_ner', None),
            'bleu_translate': results.get('bleu_translate', None),
            'total_samples': results['total_samples'],
            'valid_json_count': results.get('valid_json_count', None)
        }
        summary_data.append(row)

# Crea DataFrame riassuntivo
summary_df = pd.DataFrame(summary_data)

# Salva i risultati
output_file = f"{BASE_PATH}/evaluation_summary_invert_{MODEL_NAME_CLEAN}_6tasks.csv"
summary_df.to_csv(output_file, index=False)

print(f"✓ Risultati salvati in: {output_file}")

# Mostra tabella riassuntiva
print("\n📋 TABELLA RIASSUNTIVA:")
print(summary_df.to_string(index=False))

print("\n✅ Script di valutazione completato!")

SALVATAGGIO RISULTATI
✓ Risultati salvati in: ../Output Performance Degradation Analysis/granite3.1-moe_3b_six_task_invert/evaluation_summary_invert_granite3.1-moe_3b_6tasks.csv

📋 TABELLA RIASSUNTIVA:
  step  task_number                                    task_name  json_em  bleu_review  em_sentiment  em_emotion  em_topic    f1_ner  bleu_translate  total_samples  valid_json_count
Step 1            1                                    json_only     98.0    51.419849           NaN         NaN       NaN       NaN             NaN            500               490
Step 1            2                               json_sentiment     98.4    66.910743           NaN         NaN       NaN       NaN             NaN            500               492
Step 1            3                       json_sentiment_emotion     99.0    76.406361           NaN         NaN       NaN       NaN             NaN            500               495
Step 1            4                 json_sentiment_emotion_topic     9

In [30]:
def create_performance_visualizations(step1_results, step2_results, step3_results, 
                                    step4_results, step5_results, step6_results, 
                                    MODEL_NAME_CLEAN, BASE_PATH):
    """
    Crea visualizzazioni per l'analisi delle performance degradation.
    """
    
    plt.style.use('default')
    sns.set_palette("husl")
    
    # Colori consistenti per le metriche
    colors = {
        'json_em': '#1f77b4',      # Blu
        'bleu_translate': '#ff7f0e', # Arancione
        'em_sentiment': '#2ca02c',   # Verde
        'em_emotion': '#d62728',     # Rosso
        'em_topic': '#9467bd',       # Viola
        'f1_ner': '#8c564b'          # Marrone
    }
    
    # Funzione helper per ottenere i dati dalle diverse step
    def get_metric_data(metric_name, task_range):
        """Estrae i dati di una metrica specifica attraverso tutti i task applicabili"""
        tasks = []
        values = []
        
        # Mapping delle metriche ai task secondo la logica corretta per ordine invertito
        metric_task_mapping = {
            'json_em': range(1, 7),        # Tutti i task
            'em_sentiment': range(2, 7),   # Task 2-6
            'em_emotion': range(3, 7),     # Task 3-6  
            'em_topic': range(4, 7),       # Task 4-6
            'f1_ner': range(5, 7),         # Task 5-6
            'bleu_translate': [6]          # Solo task 6
        }
        
        for task in task_range:
            # Controlla se la metrica è applicabile a questo task
            if metric_name in metric_task_mapping and task not in metric_task_mapping[metric_name]:
                continue
                
            value = None
            
            # Cerca il valore nei vari step results, partendo dal più completo
            for step_dict in [step6_results, step5_results, step4_results, step3_results, step2_results, step1_results]:
                if task in step_dict and metric_name in step_dict[task] and step_dict[task][metric_name] is not None:
                    value = step_dict[task][metric_name]
                    break
            
            if value is not None:
                tasks.append(task)
                values.append(value)
        
        return tasks, values
    
    # 1. GRAFICO PRINCIPALE: Performance Degradation Overview
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Performance Degradation Analysis - {MODEL_NAME_CLEAN} (Inverted Order)', fontsize=16, fontweight='bold')
    
    # Subplot 1: JSON Exact Match
    ax = axes[0, 0]
    json_tasks, json_values = get_metric_data('json_em', range(1, 7))
    if json_values:
        ax.plot(json_tasks, json_values, 'o-', color=colors['json_em'], linewidth=2, markersize=8)
        ax.set_title('JSON Exact Match (%)', fontweight='bold')
        ax.set_xlabel('Task Number')
        ax.set_ylabel('Accuracy (%)')
        ax.grid(True, alpha=0.3)
        min_val, max_val = min(json_values), max(json_values)
        padding = max(5, (max_val - min_val) * 0.1) if max_val > min_val else 5
        ax.set_ylim(max(0, min_val - padding), min(100, max_val + padding))
        for i, (t, v) in enumerate(zip(json_tasks, json_values)):
            ax.annotate(f'{v:.1f}%', (t, v), textcoords="offset points", xytext=(0,10), ha='center')
    else:
        ax.text(0.5, 0.5, 'No JSON data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('JSON Exact Match (%)', fontweight='bold')
    
    # Subplot 2: BLEU Translation Score (solo Task 6)
    ax = axes[0, 1]
    bleu_tasks, bleu_values = get_metric_data('bleu_translate', [6])  
    if bleu_values:
        ax.bar(bleu_tasks, bleu_values, color=colors['bleu_translate'], width=0.5, alpha=0.8)
        ax.set_title('BLEU Translation Score (%)', fontweight='bold')
        ax.set_xlabel('Task Number')
        ax.set_ylabel('BLEU Score (%)')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, max(100, max(bleu_values) + 10))
        for i, (t, v) in enumerate(zip(bleu_tasks, bleu_values)):
            ax.annotate(f'{v:.1f}%', (t, v), textcoords="offset points", xytext=(0,10), ha='center')
    else:
        ax.text(0.5, 0.5, 'No BLEU data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('BLEU Translation Score (%)', fontweight='bold')
    
    # Subplot 3: Sentiment Analysis (Task 2-6)
    ax = axes[0, 2]
    sent_tasks, sent_values = get_metric_data('em_sentiment', range(2, 7))  
    if sent_values:
        ax.plot(sent_tasks, sent_values, '^-', color=colors['em_sentiment'], linewidth=2, markersize=8)
        ax.set_title('Sentiment Classification (%)', fontweight='bold')
        ax.set_xlabel('Task Number')
        ax.set_ylabel('Exact Match (%)')
        ax.grid(True, alpha=0.3)
        min_val, max_val = min(sent_values), max(sent_values)
        padding = max(5, (max_val - min_val) * 0.1) if max_val > min_val else 5
        ax.set_ylim(max(0, min_val - padding), min(100, max_val + padding))
        for i, (t, v) in enumerate(zip(sent_tasks, sent_values)):
            ax.annotate(f'{v:.1f}%', (t, v), textcoords="offset points", xytext=(0,10), ha='center')
    else:
        ax.text(0.5, 0.5, 'No Sentiment data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Sentiment Classification (%)', fontweight='bold')
    
    # Subplot 4: Emotion Recognition (Task 3-6)
    ax = axes[1, 0]
    emo_tasks, emo_values = get_metric_data('em_emotion', range(3, 7))  
    if emo_values:
        ax.plot(emo_tasks, emo_values, 'D-', color=colors['em_emotion'], linewidth=2, markersize=8)
        ax.set_title('Emotion Classification (%)', fontweight='bold')
        ax.set_xlabel('Task Number')
        ax.set_ylabel('Exact Match (%)')
        ax.grid(True, alpha=0.3)
        min_val, max_val = min(emo_values), max(emo_values)
        padding = max(2, (max_val - min_val) * 0.1) if max_val > min_val else 2
        ax.set_ylim(max(0, min_val - padding), max(100, max_val + padding))
        for i, (t, v) in enumerate(zip(emo_tasks, emo_values)):
            ax.annotate(f'{v:.1f}%', (t, v), textcoords="offset points", xytext=(0,10), ha='center')
    else:
        ax.text(0.5, 0.5, 'No Emotion data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Emotion Classification (%)', fontweight='bold')
    
    # Subplot 5: Topic Classification (Task 4-6)
    ax = axes[1, 1]
    topic_tasks, topic_values = get_metric_data('em_topic', range(4, 7))  
    if topic_values:
        ax.plot(topic_tasks, topic_values, 'v-', color=colors['em_topic'], linewidth=2, markersize=8)
        ax.set_title('Topic Classification (%)', fontweight='bold')
        ax.set_xlabel('Task Number')
        ax.set_ylabel('Exact Match (%)')
        ax.grid(True, alpha=0.3)
        min_val, max_val = min(topic_values), max(topic_values)
        padding = max(2, (max_val - min_val) * 0.1) if max_val > min_val else 2
        ax.set_ylim(max(0, min_val - padding), max(100, max_val + padding))
        for i, (t, v) in enumerate(zip(topic_tasks, topic_values)):
            ax.annotate(f'{v:.1f}%', (t, v), textcoords="offset points", xytext=(0,10), ha='center')
    else:
        ax.text(0.5, 0.5, 'No Topic data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Topic Classification (%)', fontweight='bold')
    
    # Subplot 6: NER F1 Score (Task 5-6)
    ax = axes[1, 2]
    ner_tasks, ner_values = get_metric_data('f1_ner', range(5, 7))  
    if ner_values:
        ax.bar(ner_tasks, ner_values, color=colors['f1_ner'], width=0.5, alpha=0.8)
        ax.set_title('Named Entity Recognition F1 (%)', fontweight='bold')
        ax.set_xlabel('Task Number')
        ax.set_ylabel('F1 Score (%)')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, max(100, max(ner_values) + 10))
        for i, (t, v) in enumerate(zip(ner_tasks, ner_values)):
            ax.annotate(f'{v:.1f}%', (t, v), textcoords="offset points", xytext=(0,10), ha='center')
    else:
        ax.text(0.5, 0.5, 'No NER data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Named Entity Recognition F1 (%)', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{BASE_PATH}/performance_degradation_overview_{MODEL_NAME_CLEAN}.png', 
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    # 2. COMPARATIVE BAR CHART: Final Performance per Task - SENZA ETICHETTE TASK
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Definizione delle metriche per ogni task (nell'ordine invertito richiesto)
    task_metrics = {
        1: [('json_em', 'JSON EM')],
        2: [('json_em', 'JSON EM'), ('em_sentiment', 'Sentiment EM')],
        3: [('json_em', 'JSON EM'), ('em_sentiment', 'Sentiment EM'), ('em_emotion', 'Emotion EM')],
        4: [('json_em', 'JSON EM'), ('em_sentiment', 'Sentiment EM'), ('em_emotion', 'Emotion EM'), ('em_topic', 'Topic EM')],
        5: [('json_em', 'JSON EM'), ('em_sentiment', 'Sentiment EM'), ('em_emotion', 'Emotion EM'), ('em_topic', 'Topic EM'), ('f1_ner', 'NER F1')],
        6: [('json_em', 'JSON EM'), ('em_sentiment', 'Sentiment EM'), ('em_emotion', 'Emotion EM'), ('em_topic', 'Topic EM'), ('f1_ner', 'NER F1'), ('bleu_translate', 'BLEU Trans')]
    }
    
    # Calcola i dati per ogni task usando DIRETTAMENTE i task_results
    all_data = []
    all_labels = []
    all_colors = []
    all_positions = []
    
    current_position = 0
    bar_width = 0.6
    group_spacing = 1.5
    
    for task in range(1, 7):
        # Ottieni i dati per questo task
        df = task_results[task]
        if df.empty:
            current_position += len(task_metrics[task]) * bar_width + group_spacing
            continue
            
        valid_json_df = df[df['json_valid'] == 1]
        
        task_values = []
        task_colors = []
        task_labels = []
        
        for metric_key, metric_label in task_metrics[task]:
            if metric_key == 'json_em':
                value = df['json_valid'].mean() * 100
            elif metric_key == 'bleu_translate' and task == 6:
                value = valid_json_df['bleu_translate'].mean() if len(valid_json_df) > 0 else 0
            elif metric_key == 'em_sentiment' and task >= 2:
                value = valid_json_df['em_sentiment'].mean() * 100 if len(valid_json_df) > 0 else 0
            elif metric_key == 'em_emotion' and task >= 3:
                value = valid_json_df['em_emotion'].mean() * 100 if len(valid_json_df) > 0 else 0
            elif metric_key == 'em_topic' and task >= 4:
                value = valid_json_df['em_topic'].mean() * 100 if len(valid_json_df) > 0 else 0
            elif metric_key == 'f1_ner' and task >= 5:
                value = valid_json_df['f1_ner'].mean() if len(valid_json_df) > 0 else 0
            else:
                value = 0
            
            task_values.append(value)
            task_colors.append(colors[metric_key])
            task_labels.append(metric_label)
        
        # Calcola posizioni per questo task
        task_positions = [current_position + i * bar_width for i in range(len(task_values))]
        
        # Aggiungi ai dati globali
        all_data.extend(task_values)
        all_colors.extend(task_colors)
        all_labels.extend([f"Task {task}\n{label}" for label in task_labels])
        all_positions.extend(task_positions)
        
        # Aggiungi separatore visuale
        current_position += len(task_metrics[task]) * bar_width + group_spacing
    
    # Crea il grafico
    bars = ax.bar(all_positions, all_data, width=bar_width, color=all_colors, alpha=0.8)
    
    # Aggiungi valori sulle barre
    for bar, value in zip(bars, all_data):
        if value > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                   f'{value:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Formattazione asse x
    ax.set_xticks(all_positions)
    ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=9)
    
    # Aggiungi linee di separazione tra i task
    separator_positions = []
    current_pos = 0
    for task in range(1, 7):
        if task > 1:  # Non aggiungere separatore prima del primo task
            separator_positions.append(current_pos - group_spacing/2)
        current_pos += len(task_metrics[task]) * bar_width + group_spacing
    
    for sep_pos in separator_positions:
        ax.axvline(x=sep_pos, color='gray', linestyle='--', alpha=0.5)
    
    # Formattazione del grafico - SENZA etichette task sopra
    ax.set_title(f'Final Performance Comparison Across Tasks - {MODEL_NAME_CLEAN} (Inverted Order)', 
                fontsize=14, fontweight='bold')
    ax.set_ylabel('Performance (%)', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 110)
    ax.set_xlabel('')
    
    plt.tight_layout()
    plt.savefig(f'{BASE_PATH}/final_performance_comparison_{MODEL_NAME_CLEAN}.png', 
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"✅ Visualizzazioni salvate:")
    print(f"   📊 {BASE_PATH}/performance_degradation_overview_{MODEL_NAME_CLEAN}.png")
    print(f"   📈 {BASE_PATH}/final_performance_comparison_{MODEL_NAME_CLEAN}.png")

# GENERAZIONE VISUALIZZAZIONI
print("=" * 60)
print("GENERAZIONE VISUALIZZAZIONI")
print("=" * 60)

# Crea le visualizzazioni
create_performance_visualizations(
    step1_results, step2_results, step3_results, 
    step4_results, step5_results, step6_results, 
    MODEL_NAME_CLEAN, BASE_PATH
)

print("\n✅ Script di valutazione completato!")

GENERAZIONE VISUALIZZAZIONI CON LAYOUT A RIGHE
✅ Visualizzazioni con layout a righe salvate:
   📊 ../Output Performance Degradation Analysis/granite3.1-moe_3b_six_task_invert/performance_degradation_overview_rows_granite3.1-moe_3b.png
   📈 ../Output Performance Degradation Analysis/granite3.1-moe_3b_six_task_invert/final_performance_comparison_rows_granite3.1-moe_3b.png

✅ Script di valutazione con layout a righe completato!


TypeError: create_performance_visualizations() missing 1 required positional argument: 'task_results'